# 11 — Advanced Feature Engineering Patterns (Solutions)

## Problem Definition
Design high-capacity feature maps while controlling dimensionality, variance, and interpretability.

## Required Prior Knowledge
- Notebook 10 concepts: standardization, conditional expectations, leakage-safe estimators.
- Multivariate polynomial notation and combinatorics basics.

## New Concepts Introduced
- Polynomial basis expansion in $\mathbb{R}^d$.
- Interaction terms as tensor products.
- Curse of dimensionality counting argument.
- Aggregation features as conditional expectations.
- SHAP from Shapley values with explicit combinatorial weights.

### Mathematical Foundations
For $x\\in\\mathbb{R}^d$, the degree-$p$ polynomial basis includes monomials
$$
x_1^{\\alpha_1}\\cdots x_d^{\\alpha_d},\\qquad
\\alpha_j\\in\\mathbb{N}_0,\\quad \\sum_{j=1}^d \\alpha_j \\le p.
$$
Number of monomials:
$$
N(d,p)=\\binom{d+p}{p}.
$$

**Tensor-product interaction (pairwise):**
$$
\\psi_{ij}(x)=x_i x_j.
$$

**Aggregation as conditional expectation:**
$$
g(x)=\\mathbb{E}[Y\\mid G(x)],
$$
where $G(x)$ maps samples into groups.

**Shapley value for feature $j$:**
$$
\\phi_j(v)=\\sum_{S\\subseteq N\\setminus\\{j\\}}
\\frac{|S|!(M-|S|-1)!}{M!}\\Big(v(S\\cup\\{j\\})-v(S)\\Big).
$$

## Symbol-by-Symbol Explanation
| Symbol | Meaning |
|---|---|
| $x_i$ | feature vector for sample $i$ |
| $y_i$ | target for sample $i$ |
| $f_\theta$ | model parameterized by $\theta$ |
| $L(\cdot,\cdot)$ | per-sample loss function |
| $n$ | number of samples |
| $d$ | raw feature dimension |
| $p$ | transformed feature dimension / polynomial degree context |
| $K$ | number of folds / partitions |
| $V_k$ | validation index set for fold $k$ |
| $\lambda,\alpha,\tau$ | regularization/smoothing/confidence hyperparameters |

### Step-by-Step Derivation
1. Count degree-exact-$k$ monomials by stars-and-bars:
   $$
   \\#\\{\\alpha: \\sum_j \\alpha_j=k\\}=\\binom{d+k-1}{k}.
   $$
2. Sum $k=0$ to $p$:
   $$
   N(d,p)=\\sum_{k=0}^{p}\\binom{d+k-1}{k}=\\binom{d+p}{p}.
   $$
3. Curse-of-dimensionality consequence: if $d=50, p=3$, then
   $$
   N(50,3)=\\binom{53}{3}=23426,
   $$
   which can exceed sample-efficient regimes.
4. Shapley combinatorial weight is the fraction of feature-order permutations where subset $S$ appears before feature $j$:
   $$
   w(S)=\\frac{|S|!(M-|S|-1)!}{M!}.
   $$
5. Weighted marginal contributions sum to total attribution and satisfy efficiency:
   $$
   \\sum_{j=1}^M \\phi_j = v(N)-v(\\varnothing).
   $$

## Intuition
Polynomial and interaction features increase expressive power; Shapley values distribute model output contribution fairly across features with permutation-based weighting.

## Mapping from Math to Implementation
- `polynomial_basis` builds monomial expansion.
- `add_interaction_columns` creates selected tensor-product terms.
- Aggregation features are implemented through smoothed conditional expectation.
- SHAP derivation is validated with exact enumeration on a tiny cooperative game.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn

from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier

try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False

from feature_utils import (
    set_global_seed,
    standardize_train_valid,
    monotone_log1p,
    one_hot_basis,
    polynomial_basis,
    add_interaction_columns,
    target_encode_oof,
    conditional_expectation_feature,
)
from cv_utils import (
    empirical_risk,
    make_splitter,
    oof_cv_predictions,
    cv_bias_variance_decomposition,
    leakage_inflation,
    simulate_public_private_variance,
)
from ensemble_utils import (
    blend_predictions,
    bagging_variance_formula,
    ensemble_variance_from_covariance,
    oof_stacking,
    stacking_predict,
)
from experiment_logger import ExperimentLogger, ExperimentRecord

SEED = 42
set_global_seed(SEED)

MODULE_DIR = Path('.').resolve()

## Synthetic Experiment

In [ ]:
from sklearn.datasets import make_regression

X_syn, y_syn = make_regression(n_samples=2500, n_features=6, noise=18.0, random_state=SEED)
X_df = pd.DataFrame(X_syn, columns=[f"f{i}" for i in range(X_syn.shape[1])])

poly_X, poly = polynomial_basis(X_df.values, degree=2, include_bias=False)
base_rmse = np.sqrt(mean_squared_error(y_syn, LinearRegression().fit(X_df.values, y_syn).predict(X_df.values)))
poly_rmse = np.sqrt(mean_squared_error(y_syn, Ridge(alpha=2.0).fit(poly_X, y_syn).predict(poly_X)))

print({"poly_feature_count": poly_X.shape[1], "base_fit_rmse": base_rmse, "poly_fit_rmse": poly_rmse})

## Real Dataset Experiment

In [ ]:
bc = load_breast_cancer(as_frame=True)
X_real = bc.data.copy()
y_real = bc.target.to_numpy(dtype=int)

X_small = X_real.iloc[:, :8].copy()
X_small = add_interaction_columns(X_small, [("mean radius", "mean texture"), ("mean area", "mean smoothness")])
X_train, X_valid, y_train, y_valid = train_test_split(X_small, y_real, test_size=0.25, random_state=SEED, stratify=y_real)

scaler = StandardScaler().fit(X_train)
Xt = scaler.transform(X_train)
Xv = scaler.transform(X_valid)

clf = LogisticRegression(max_iter=2000).fit(Xt, y_train)
proba = clf.predict_proba(Xv)[:, 1]
auc = roc_auc_score(y_valid, proba)
print({"breast_cancer_auc": auc, "n_features_after_interactions": X_small.shape[1]})

## Diagnostic Analysis

In [ ]:
# Exact Shapley demonstration on 3-feature toy value function.
import itertools
from math import factorial

def v(S):
    S = set(S)
    score = 0.0
    if "a" in S: score += 2.0
    if "b" in S: score += 1.0
    if "c" in S: score += 1.5
    if "a" in S and "b" in S: score += 0.8
    return score

features = ["a", "b", "c"]
M = len(features)
phi = {}
for j in features:
    others = [f for f in features if f != j]
    contrib = 0.0
    for r in range(len(others) + 1):
        for subset in itertools.combinations(others, r):
            w = factorial(len(subset)) * factorial(M - len(subset) - 1) / factorial(M)
            contrib += w * (v(set(subset) | {j}) - v(set(subset)))
    phi[j] = contrib

print("Shapley values:", phi, "sum=", sum(phi.values()), "total=", v(features) - v(set()))

## Failure Case Demonstration

In [ ]:
# Failure case: uncontrolled high-degree expansion creates severe overfitting.
poly_high, _ = polynomial_basis(X_df.values, degree=4, include_bias=False)
model_overfit = LinearRegression().fit(poly_high, y_syn)
pred_overfit = model_overfit.predict(poly_high)
rmse_overfit = np.sqrt(mean_squared_error(y_syn, pred_overfit))
print({"degree4_feature_count": poly_high.shape[1], "apparent_train_rmse": rmse_overfit})

## Exercise Ladder (basic → advanced → research-level)
1. Derive the feature count formula for degree-exact-$p$ without the summation step.
2. Prove Shapley efficiency and symmetry axioms for the 3-feature case.
3. Show when aggregation features reduce variance but increase bias.
4. Build a sparse polynomial map and compare with dense expansion.

## Solution Notes
- Verify deterministic behavior by re-running all cells with the same seed and matching key metrics.
- Confirm that no fold-level preprocessing leaks validation targets/features into training statistics.
- Compare synthetic-vs-real conclusions and report where assumptions diverge.

## Summary of Mathematical Insights
Advanced feature maps expand hypothesis space; controlling dimension growth and attribution discipline is necessary to keep improvements generalizable.